In [1]:
# -*- coding: utf-8 -*-

# Phase 3 — 화주 세그먼테이션 (v4)

반복 발주 화주(컨설팅 타깃)와 스팟 화주를 분리한다.

**v3 대비 변경**

- v2에서는 원본 로그의 화주ID가 무작위라 `목업재생성_화주구조`로 화주 축을 사후에
  심어줘야 했다. **v4는 생성 단계에서 관성 로직(화주별 템플릿 복사)으로 구조를 만든다.**
  그 재배정 노트북은 `_보관_v3이전/`으로 뺐다.
- 화주 70명 → **300명**. 군집 실루엣과 규칙 판정이 훨씬 안정적이다.
- 컬럼: `물량_파렛트`→`파렛트`, `출발지`→`노선ID`
- 3-4 복원검증의 정답이 바뀌었다. v4 `화주프로파일`에는 `화주유형`·`EOQ_T`가 없고
  대신 **템플릿(주노선·주품목·기본요일·기본창)과 성향(리드성향·유연성향·변형률)**이 있다.
  그래서 "라벨을 맞히는가"가 아니라 **"심어둔 템플릿을 되찾는가"**를 검증한다.

In [2]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from scipy.signal import periodogram
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score

load_dotenv()
SRC = Path(os.getenv("DATA_DIR", ".")).expanduser().resolve()
OUT = SRC / "out"; OUT.mkdir(exist_ok=True)
XLSX = SRC / os.getenv("DATA_FILE", "유연오더_가상데이터_v13.xlsx")

df = pd.read_csv(OUT / "phase1_분석테이블.csv", parse_dates=["등록일시", "상차희망"])
df["등록일"] = df["등록일시"].dt.normalize()
print(f"[load] {len(df)}건 / 화주 {df['화주ID'].nunique()}명")

[load] 12000건 / 화주 300명


## 3-1. 화주별 피처 집계

발주 간격·요일 집중도·물량 변동성이 규칙성의 지표다.
노선 집중도는 v2의 `출발지` 대신 `노선ID`로 계산한다(v4는 O-D 쌍이 노선으로 묶여 있다).

In [3]:
def agg(g):
    d = g.sort_values("등록일시")
    gaps = d["등록일"].diff().dt.days.dropna()
    wd = d["상차희망"].dt.weekday          # 관성은 '상차 요일'에 새겨진다
    vol = d["파렛트"]
    wk = (d["등록일"].dt.isocalendar().year.astype(str) + "-"
          + d["등록일"].dt.isocalendar().week.astype(str))
    active_weeks = pd.to_datetime(d.groupby(wk)["등록일"].first()).sort_values()
    week_gaps = active_weeks.diff().dt.days.dropna()
    return pd.Series({
        "발주건수": len(d),
        "관측기간_일": (d["등록일"].max() - d["등록일"].min()).days,
        "발주간격_최빈": gaps.mode().iloc[0] if len(gaps) else np.nan,
        "발주간격_평균": gaps.mean() if len(gaps) else np.nan,
        "발주간격_중앙": gaps.median() if len(gaps) else np.nan,
        "발주간격_std": gaps.std() if len(gaps) > 1 else np.nan,
        "활동주수": len(active_weeks),
        "주간격_중앙": week_gaps.median() if len(week_gaps) else np.nan,
        "요일집중도": wd.value_counts().max() / len(d),
        "물량_CV": vol.std() / vol.mean() if len(d) > 1 and vol.mean() else np.nan,
        "평균_유연성": d["유연성지수"].mean(),
        "평균_리드타임": d["리드타임_h"].mean(),
        "노선_집중도": d["노선ID"].value_counts().max() / len(d),
        "품목_집중도": d["품목"].value_counts().max() / len(d),
        "주노선_실측": d["노선ID"].value_counts().index[0],
        "주품목_실측": d["품목"].value_counts().index[0],
        "기본요일_실측": int(wd.value_counts().index[0]),
        "긴급비율": (d["긴급여부"] == "긴급").mean(),
        "미승인비율": (d["원화주_조정권한_증빙"].astype(str) == "미승인").mean(),
        "유찰률": d["유찰"].mean(),
        "평균_체결배율": d["체결배율"].mean(),
    })


S = df.groupby("화주ID").apply(agg, include_groups=False)
S["간격_CV"] = S["발주간격_std"] / S["발주간격_평균"]
print(f"[집계] 화주 {len(S)}명 × 피처 {S.shape[1]}개")

[집계] 화주 300명 × 피처 22개


## 3-2. 분류 — 규칙 우선(해석 가능성), 군집 보조

관측수가 적으면 요일집중도가 기계적으로 1.0에 가까워진다(n=2면 최소 0.5).
규칙 적용 전에 최소 관측수 조건을 먼저 건다.

In [4]:
MIN_N = 3


def rule_v1(r):
    """원안 — 요일집중도 ≥0.7 & 간격 최빈 7±1"""
    if r["발주건수"] < MIN_N:
        return "관측부족"
    g = r["발주간격_최빈"]
    if r["요일집중도"] >= 0.7 and 6 <= g <= 8:
        return "고정주간"
    if 13 <= g <= 15:
        return "격주"
    if r["발주간격_평균"] <= 5.5 and r["발주건수"] >= 8:
        return "고빈도"
    return "스팟"


def rule_v2(r):
    """개선안 — 원안의 약점 보완 + 배치 주문 대응
       ① 간격 최빈값은 발주일이 ±1일만 흔들려도 무너진다 → 중앙값으로 교체
       ② 요일집중도 0.7은 n이 작을 때 과도 → 0.6으로 완화
       ③ 관측부족(n<3)은 별도 범주가 아니라 스팟으로 흡수
       ④ 같은 주에 여러 건을 몰아 등록하는 배치 패턴이 있으면 등록일 간격이
          기계적으로 짧아져 매주 발주자가 고빈도로 오분류된다 → 간격 판정을
          등록일 단위가 아니라 활동한 주(週) 단위로 계산하고, 요일 집중 여부를
          고빈도보다 먼저 확인한다
    """
    if r["발주건수"] < MIN_N:
        return "스팟"
    if r["요일집중도"] >= 0.6 and 6 <= r["주간격_중앙"] <= 8:
        return "고정주간"
    if r["발주건수"] >= 8 and r["발주간격_평균"] <= 5.5:
        return "고빈도"
    if 12.5 <= r["주간격_중앙"] <= 15.5:
        return "격주"
    return "스팟"


S["세그먼트_원안"] = S.apply(rule_v1, axis=1)
S["세그먼트_규칙"] = S.apply(rule_v2, axis=1)
seg_summary = (S.groupby("세그먼트_규칙")
               .agg(화주수=("발주건수", "size"), 총발주=("발주건수", "sum"),
                    평균간격=("발주간격_평균", "mean"), 요일집중도=("요일집중도", "mean"),
                    평균유연성=("평균_유연성", "mean"), 노선집중도=("노선_집중도", "mean"),
                    유찰률=("유찰률", "mean"))
               .round(3).sort_values("총발주", ascending=False))
print("\n[3-2 규칙 기반 세그먼트]")
print(seg_summary.to_string())


[3-2 규칙 기반 세그먼트]
         화주수   총발주   평균간격  요일집중도  평균유연성  노선집중도    유찰률
세그먼트_규칙                                              
고정주간     189  7790  5.667  0.818  0.344  0.792  0.095
스팟        74  2587  6.648  0.661  0.326  0.769  0.073
고빈도       34  1526  5.059  0.559  0.354  0.723  0.082
격주         3    97  7.355  0.794  0.455  0.754  0.063


In [5]:
# ── 군집 교차 확인 ─────────────────────────────────────────────
FEATS = ["발주건수", "발주간격_평균", "간격_CV", "요일집중도",
         "물량_CV", "평균_유연성", "노선_집중도"]
X = S[FEATS].copy()
X = X.fillna(X.median())
Z = StandardScaler().fit_transform(X)

print("\n[군집 교차 확인] K-means")
sil = {}
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=20, random_state=42).fit(Z)
    sil[k] = silhouette_score(Z, km.labels_)
    print(f"   k={k}  실루엣={sil[k]:.3f}")
BEST_K = max(sil, key=sil.get)
km = KMeans(n_clusters=BEST_K, n_init=20, random_state=42).fit(Z)
S["군집_kmeans"] = km.labels_
print(f"   → 최적 k={BEST_K} (실루엣 {sil[BEST_K]:.3f})")
if sil[BEST_K] < 0.25:
    print("   ※ 실루엣 0.25 미만 — 군집 구조가 약하다. 규칙 기반 분류를 우선한다.")

db = DBSCAN(eps=1.2, min_samples=5).fit(Z)
S["군집_dbscan"] = db.labels_
n_db = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
print(f"[군집 교차 확인] DBSCAN: 군집 {n_db}개, 노이즈 {int((db.labels_ == -1).sum())}명")

print(f"\n규칙 vs K-means 일치도(ARI) = {adjusted_rand_score(S['세그먼트_규칙'], S['군집_kmeans']):.3f}")
print(pd.crosstab(S["세그먼트_규칙"], S["군집_kmeans"]).to_string())


[군집 교차 확인] K-means
   k=2  실루엣=0.166
   k=3  실루엣=0.144
   k=4  실루엣=0.142
   k=5  실루엣=0.150
   k=6  실루엣=0.136
   → 최적 k=2 (실루엣 0.166)
   ※ 실루엣 0.25 미만 — 군집 구조가 약하다. 규칙 기반 분류를 우선한다.
[군집 교차 확인] DBSCAN: 군집 5개, 노이즈 222명

규칙 vs K-means 일치도(ARI) = 0.123
군집_kmeans    0   1
세그먼트_규칙           
격주           0   3
고빈도         34   0
고정주간       121  68
스팟          13  61


## 3-3. 주기성 검정 (보조)

In [6]:
print("\n[3-3 주기성 검정]")
daily = df.groupby("등록일").size()
idx = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
daily = daily.reindex(idx, fill_value=0)
f, P = periodogram(daily.values - daily.values.mean())
with np.errstate(divide="ignore"):
    per = 1 / f
top = pd.DataFrame({"주기_일": per, "파워": P}).replace([np.inf], np.nan).dropna()
top = top[(top["주기_일"] >= 2) & (top["주기_일"] <= 30)].nlargest(5, "파워")
print("  전체 일별 발주량 주기도 상위:")
print(top.round(2).to_string(index=False))

x = daily.values.astype(float) - daily.values.mean()
ci = 1.96 / np.sqrt(len(x))
acf = {k: np.corrcoef(x[:-k], x[k:])[0, 1] for k in range(1, 16)}
sig = [k for k, a in acf.items() if abs(a) > ci]
print(f"  ACF 유의 lag(±{ci:.3f}): {sig if sig else '없음'}  | lag7={acf[7]:+.3f}, lag14={acf[14]:+.3f}")

for seg in ["고정주간", "격주"]:
    ids = S.index[S["세그먼트_규칙"] == seg]
    if len(ids) == 0:
        print(f"  [{seg}] 해당 화주 없음")
        continue
    sub = df[df["화주ID"].isin(ids)].groupby("등록일").size().reindex(idx, fill_value=0)
    xs = sub.values.astype(float) - sub.values.mean()
    print(f"  [{seg}] 화주 {len(ids)}명 집계: "
          f"ACF lag7={np.corrcoef(xs[:-7], xs[7:])[0,1]:+.3f}, "
          f"lag14={np.corrcoef(xs[:-14], xs[14:])[0,1]:+.3f}")


[3-3 주기성 검정]
  전체 일별 발주량 주기도 상위:
 주기_일       파워
 6.94 71757.15
 7.15 17039.05
 3.52  3548.32
 7.36  3032.36
 6.75  2190.65
  ACF 유의 lag(±0.126): [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]  | lag7=+0.841, lag14=+0.845
  [고정주간] 화주 189명 집계: ACF lag7=+0.832, lag14=+0.824
  [격주] 화주 3명 집계: ACF lag7=+0.119, lag14=+0.197


## 3-4. 복원 검증 — 생성기가 심어둔 템플릿을 되찾는가

`화주프로파일` 시트는 **생성 파라미터 정답지**다. 모델 입력에는 절대 넣지 않지만,
파이프라인이 구조를 복원하는지 채점하는 용도로는 쓸 수 있다.

> **이 정확도를 발표에서 '분류 정확도'로 단독 인용하면 안 된다.**
> 현실 예측력이 아니라 "설계한 구조를 집계가 되찾는가"의 자기검증이다.

In [7]:
try:
    truth = pd.read_excel(XLSX, sheet_name="화주프로파일").set_index("화주ID")
    S = S.join(truth[["주노선", "주품목", "기본요일", "기본창", "리드성향", "유연성향", "변형률"]])

    hit_route = (S["주노선_실측"] == S["주노선"]).mean()
    hit_item = (S["주품목_실측"] == S["주품목"]).mean()
    hit_dow = (S["기본요일_실측"] == S["기본요일"]).mean()
    print(f"\n[3-4] 템플릿 복원율 — 주노선 {hit_route*100:.1f}% / "
          f"주품목 {hit_item*100:.1f}% / 기본요일 {hit_dow*100:.1f}%")
    print("   생성기 변형률이 12~28%이므로 70~88% 대역이 정상. 100%면 관성이 과하다는 뜻.")

    for a, b, nm in [("평균_유연성", "유연성향", "유연성"),
                     ("평균_리드타임", "리드성향", "리드타임"),
                     ("요일집중도", "변형률", "요일집중도 vs 변형률(음의 상관이어야 정상)")]:
        s = S[[a, b]].dropna()
        print(f"   {nm:36s} r = {np.corrcoef(s[a], s[b])[0,1]:+.3f}")

    # 세그먼트가 실제 발주 강도와 대응하는가
    print("\n[세그먼트 × 생성 파라미터]")
    print(S.groupby("세그먼트_규칙")[["유연성향", "리드성향", "변형률", "발주건수"]]
          .mean().round(3).to_string())
except Exception as e:
    print(f"\n[3-4] 정답 라벨 없음 — 복원 검증 생략 ({e})")


[3-4] 템플릿 복원율 — 주노선 95.7% / 주품목 97.0% / 기본요일 92.7%
   생성기 변형률이 12~28%이므로 70~88% 대역이 정상. 100%면 관성이 과하다는 뜻.
   유연성                                  r = +0.397
   리드타임                                 r = +0.964
   요일집중도 vs 변형률(음의 상관이어야 정상)            r = +0.009

[세그먼트 × 생성 파라미터]
          유연성향    리드성향    변형률    발주건수
세그먼트_규칙                              
격주       0.661  27.367  0.157  32.333
고빈도      0.387  35.179  0.196  44.882
고정주간     0.394  27.666  0.201  41.217
스팟       0.401  32.401  0.195  34.959


## 저장 — 영업 우선순위 리스트

In [8]:
PRIORITY = {"고빈도": 1, "고정주간": 2, "격주": 3, "스팟": 4}
S["영업우선순위"] = S["세그먼트_규칙"].map(PRIORITY)
S_out = S.sort_values(["영업우선순위", "평균_유연성"], ascending=[True, False])
S_out.round(4).to_csv(OUT / "phase3_화주세그먼트.csv", encoding="utf-8-sig")
seg_summary.to_csv(OUT / "phase3_세그먼트요약.csv", encoding="utf-8-sig")

print("\n[영업 타깃 상위 10 — 반복발주 × 고유연성]")
cols = ["발주건수", "발주간격_평균", "요일집중도", "평균_유연성", "노선_집중도", "주품목_실측", "세그먼트_규칙"]
print(S_out[cols].head(10).round(2).to_string())
print(f"\n[저장] {OUT}")


[영업 타깃 상위 10 — 반복발주 × 고유연성]
      발주건수  발주간격_평균  요일집중도  평균_유연성  노선_집중도 주품목_실측 세그먼트_규칙
화주ID                                                     
S127    44     5.44   0.55    0.61    0.84   전자부품     고빈도
S125    43     5.38   0.44    0.50    0.93    기계류     고빈도
S298    44     5.12   0.77    0.49    0.39   식품가공     고빈도
S126    49     4.67   0.55    0.47    0.90  자동차부품     고빈도
S180    43     5.33   0.88    0.46    0.70    철강재     고빈도
S219    43     5.36   0.72    0.45    0.79   냉동식품     고빈도
S251    41     5.28   0.49    0.45    0.44   생활용품     고빈도
S089    42     5.44   0.83    0.44    0.76   생활용품     고빈도
S119    44     5.23   0.55    0.44    0.61   전자부품     고빈도
S154    50     4.71   0.80    0.42    0.34   냉동식품     고빈도

[저장] /Volumes/SSD/공모전자료/유통_물류-해커톤/데이터셋-모음/out
